# Inheco ODTC

The Inheco ODTC (On Deck Thermal Cycler) is a thermocycler designed for automated PCR workflows. It features:

- Precise temperature control (4 -- 99 °C, up to 4.4 °C/s ramp rate)
- Heated lid to prevent condensation
- Motorized door for automated plate handling
- 96-well plate format

The ODTC communicates over Ethernet using the SiLA 2 protocol. You will need the IP address of the device and network connectivity between your computer and the ODTC.

See the [Inheco ODTC product page](https://www.inheco.com/odtc.html) for hardware details.

## Setup

In [ ]:
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.legacy.thermocycling.inheco import ExperimentalODTCBackend
from pylabrobot.legacy.thermocycling.thermocycler import Thermocycler

odtc = Thermocycler(
  name="odtc",
  size_x=159.0,
  size_y=245.0,
  size_z=228.0,
  backend=ExperimentalODTCBackend(ip="169.254.151.99"),  # replace with your ODTC's IP address
  child_location=Coordinate(0, 0, 0),
)
await odtc.setup()

## Door control

Open and close the motorized door for plate access:

In [ ]:
await odtc.open_lid()

In [ ]:
await odtc.close_lid()

## Temperature control

The ODTC exposes block and lid temperature control. For general concepts, see [Temperature Control](../../capabilities/temperature-control).

### Reading sensor data

Get current temperatures from all sensors:

In [ ]:
sensor_data = await odtc.backend.get_sensor_data()
print(sensor_data)

### Setting block temperature

Set a constant block temperature. The ODTC uses a "pre-method" internally, which takes several minutes to stabilize the block and lid evenly before reaching the target.

In [ ]:
await odtc.set_block_temperature([37.0])

In [ ]:
temp = await odtc.get_block_current_temperature()
print(f"Block temperature: {temp[0]} °C")

In [ ]:
await odtc.deactivate_block()

## Running PCR protocols

The ODTC can run multi-stage PCR protocols defined using `Protocol`, `Stage`, and `Step` objects. A protocol consists of stages, each containing steps with a target temperature and hold time. Stages can repeat multiple times for cycling.

### Defining a protocol

In [ ]:
from pylabrobot.legacy.thermocycling.standard import Protocol, Stage, Step

pcr_protocol = Protocol(
  stages=[
    # Initial denaturation
    Stage(
      steps=[Step(temperature=[95.0], hold_seconds=300)],  # 95 °C for 5 min
      repeats=1,
    ),
    # PCR cycling (30 cycles)
    Stage(
      steps=[
        Step(temperature=[95.0], hold_seconds=30),   # Denature: 95 °C, 30 s
        Step(temperature=[55.0], hold_seconds=30),   # Anneal:   55 °C, 30 s
        Step(temperature=[72.0], hold_seconds=60),   # Extend:   72 °C, 60 s
      ],
      repeats=30,
    ),
    # Final extension
    Stage(
      steps=[Step(temperature=[72.0], hold_seconds=600)],  # 72 °C for 10 min
      repeats=1,
    ),
    # Hold
    Stage(
      steps=[Step(temperature=[4.0], hold_seconds=0)],  # 4 °C hold
      repeats=1,
    ),
  ]
)

### Running the protocol

In [ ]:
await odtc.run_protocol(
  protocol=pcr_protocol,
  block_max_volume=20.0,           # maximum sample volume in uL
  start_block_temperature=25.0,    # starting block temperature
  start_lid_temperature=105.0,     # lid temperature (typically 105 °C to prevent condensation)
)

### Custom ramp rates

You can specify custom temperature ramp rates (°C/s) for individual steps via the `rate` parameter:

In [ ]:
custom_protocol = Protocol(
  stages=[
    Stage(
      steps=[
        Step(temperature=[95.0], hold_seconds=60, rate=4.4),  # fast ramp
        Step(temperature=[60.0], hold_seconds=30, rate=2.0),  # slower ramp
      ],
      repeats=1,
    ),
  ]
)

await odtc.run_protocol(
  protocol=custom_protocol,
  block_max_volume=25.0,
  start_block_temperature=25.0,
  start_lid_temperature=105.0,
)

## Teardown

In [ ]:
await odtc.stop()